In [10]:
# 이 코드는 colab에서 실행하는 경우에만 사용합니다.
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os
import numpy as np

In [12]:
# 이 코드는 colab에서 실행하는 경우에만 사용합니다.
def maybe_make_mnist_raw(out_root="/content/mnist_raw"):
    # 이미 있으면 스킵
    train_dir = os.path.join(out_root, "training", "0")
    test_dir  = os.path.join(out_root, "testing", "0")
    if os.path.isdir(train_dir) and os.path.isdir(test_dir) and len(os.listdir(train_dir)) > 0:
        print("[OK] mnist_raw already exists:", out_root)
        return out_root

    print("[INFO] Creating mnist_raw at:", out_root)
    os.makedirs(out_root, exist_ok=True)

    # TensorFlow로 다운로드만 사용
    from tensorflow.keras.datasets import mnist
    (x_train, y_train), (x_test, y_test) = mnist.load_data()  # (N,28,28), uint8

    def write_split(split_name, X, y):
        split_root = os.path.join(out_root, split_name)
        for label in range(10):
            os.makedirs(os.path.join(split_root, str(label)), exist_ok=True)

        # 파일 저장
        for i in range(X.shape[0]):
            label = int(y[i])
            fpath = os.path.join(split_root, str(label), f"{i:05d}.raw")
            X[i].astype(np.uint8).ravel().tofile(fpath)

    write_split("training", x_train, y_train)
    write_split("testing",  x_test,  y_test)

    print("[DONE] mnist_raw created.")
    return out_root

maybe_make_mnist_raw("/content/mnist_raw")


[OK] mnist_raw already exists: /content/mnist_raw


'/content/mnist_raw'

In [13]:
# 평가만 하므로 backward는 없어도 됨
class SimpleNeuralNet:
    def __init__(self, input_size, hidden_size, output_size):
        # load_weights로부터 덮어씌워짐
        self.W1 = np.zeros((hidden_size, input_size), dtype=np.float32)
        self.b1 = np.zeros((hidden_size, 1), dtype=np.float32)
        self.W2 = np.zeros((output_size, hidden_size), dtype=np.float32)
        self.b2 = np.zeros((output_size, 1), dtype=np.float32)

    def relu(self, z):
        return np.maximum(0, z)

    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)

    def forward(self, x):
        self.z1 = np.dot(self.W1, x) + self.b1
        self.a1 = self.relu(self.z1)
        
        self.z2 = np.dot(self.W2, self.a1) + self.b2
        self.a2 = self.softmax(self.z2) 
        return self.a2

In [14]:
# 데이터셋 읽어오는 함수
def load_mnist_raw(root_dir, split="training"):
    split_dir = os.path.join(root_dir, split)
    X_list, y_list = [], [] # X: 이미지(784차원), Y: 정답(label)

    print(f"[{split}] 데이터 탐색 시작: {split_dir}")

    # 디렉토리가 없는 경우
    if not os.path.exists(split_dir):
        print(f"[오류] 폴더가 없습니다: {split_dir}")
        return np.array([]), np.array([]), np.array([])

    total_loaded = 0  # 로드된 개수 카운트

    # 클래스별 디렉토리 순회
    for label in range(10):
        class_dir = os.path.join(split_dir, str(label))

        # 디렉토리가 없으면 건너뜀
        if not os.path.isdir(class_dir):
            continue

        files = os.listdir(class_dir)
        for fname in files:
            if not fname.endswith(".raw"):
                continue

            fpath = os.path.join(class_dir, fname)
            try:
                # 파일 전체를 읽음
                with open(fpath, "rb") as f:
                    data = np.frombuffer(f.read(), dtype=np.uint8)

                # 정해진 포맷(28x28)이 아닌 경우 건너뜀
                if data.size != 784:
                    continue

                # 각 화솟값(0~255)을 0~1로 정규화
                arr = data.astype(np.float32) / 255.0
                X_list.append(arr)
                y_list.append(label)

                total_loaded += 1

                # 1000개 단위로 로드 현황 알림
                if total_loaded % 1000 == 0:
                    print(f" -> {total_loaded}개 로드 완료...")

            except Exception:
                continue

    # raw 데이터가 없는 경우
    if len(X_list) == 0:
        raise ValueError(f"'{split}' 폴더에서 .raw 데이터를 찾을 수 없습니다.")

    print(f"[{split}] 총 {total_loaded}개 로드 완료!")

    X = np.stack(X_list, axis=0)    # 입력 샘플
    y_int = np.array(y_list, dtype=np.int64)    # 정답 저장
    Y = np.eye(10, dtype=np.float32)[y_int] # 정수 label을 one-hot 벡터로 변환

    # ex)
    # y_int = [1, 2, 3] -> Y = [[0,1,0,0,0,0,0,0,0,0], [0,0,1,0,0,0,0,0,0,0], [0,0,0,1,0,0,0,0,0,0]]

    return X.T, Y.T, y_int

In [15]:
def accuracy_from_net(net, X, y_int, batch_size=1000):
    correct_count = 0
    N = X.shape[1]

    for i in range(0, N, batch_size):
        X_batch = X[:, i:i + batch_size]
        y_batch = y_int[i:i + batch_size]

        out = net.forward(X_batch)
        pred = np.argmax(out, axis=0)
        correct_count += np.sum(pred == y_batch)

    return float(correct_count / N)

In [16]:
def load_weights(net, filename="weights_final.npz"):
    if not os.path.exists(filename):
        raise FileNotFoundError(f"가중치 파일이 없습니다: {filename}")

    data = np.load(filename)

    net.W1 = data["W1"]
    net.b1 = data["b1"]
    net.W2 = data["W2"]
    net.b2 = data["b2"]

    print(f"[로드] 가중치 로드 완료: {filename}")
    print(f"  W1: {net.W1.shape}, b1: {net.b1.shape}, W2: {net.W2.shape}, b2: {net.b2.shape}")

In [17]:
def main():
    # 로컬로 실행시 경로
    # current_dir = os.path.dirname(os.path.abspath(__file__)) # 현재 파일 기준 경로
    # data_root = os.path.join(current_dir, "mnist_raw") # 데이터 폴더 / 가중치 파일 경로

    # colab으로 실행시 경로
    current_dir = "/content/"

    # 데이터 폴더 / 가중치 파일 경로
    data_root = os.path.join(current_dir, "mnist_raw")
    weights_path = os.path.join(current_dir, "drive/MyDrive/weights_final_v2.npz")

    # 네트워크 크기 (학습 때와 동일해야 함)
    input_size = 784
    hidden_size = 128
    output_size = 10

    # 네트워크 생성 + 가중치 로드
    net = SimpleNeuralNet(input_size, hidden_size, output_size)
    load_weights(net, weights_path)

    # 테스트 데이터 로드 + 평가
    X_test, _, y_test_int = load_mnist_raw(data_root, "testing")

    test_acc = accuracy_from_net(net, X_test, y_test_int, batch_size=1000)
    print(f"\n[결과] Test Accuracy: {test_acc:.4f}")

    # 예측 예시(랜덤 10개)
    rng = np.random.default_rng(0)
    idx = rng.choice(X_test.shape[1], size=10, replace=False)
    out = net.forward(X_test[:, idx])
    pred = np.argmax(out, axis=0)

    print("\n=== 테스트 예측 예시(랜덤 10개) ===")
    print("선택 인덱스:", idx)
    print("예측:", pred)
    print("정답:", y_test_int[idx])

In [18]:
if __name__ == "__main__":
    main()

[로드] 가중치 로드 완료: /content/drive/MyDrive/weights_final_v2.npz
  W1: (128, 784), b1: (128, 1), W2: (10, 128), b2: (10, 1)
[testing] 데이터 탐색 시작: /content/mnist_raw/testing
 -> 1000개 로드 완료...
 -> 2000개 로드 완료...
 -> 3000개 로드 완료...
 -> 4000개 로드 완료...
 -> 5000개 로드 완료...
 -> 6000개 로드 완료...
 -> 7000개 로드 완료...
 -> 8000개 로드 완료...
 -> 9000개 로드 완료...
 -> 10000개 로드 완료...
[testing] 총 10000개 로드 완료!

[결과] Test Accuracy: 0.9810

=== 테스트 예측 예시(랜덤 10개) ===
선택 인덱스: [8498 8132 6364 5107 2696  409  165 3076 1752  752]
예측: [7 8 6 4 2 0 0 2 1 0]
정답: [8 8 6 4 2 0 0 2 1 0]
